In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load both sheets
sheets = pd.read_excel(
    "../data/raw/online_retail_II.xlsx",
    sheet_name=None
)

# Combine into one DataFrame
df = pd.concat(sheets.values(), ignore_index=True)

print("Total rows:", len(df))
print("Columns:", df.columns.tolist())

df.head()

Total rows: 1067371
Columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [2]:
# Inspect dataset
print(df.info())

# Missing values
print(df.isnull().sum())

# Number of rows in each sheet
for name, sheet in sheets.items():
    print(name, len(sheet))

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[us]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 65.1+ MB
None


Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64
Year 2009-2010 525461
Year 2010-2011 541910


## DE1 data profile

One row per column: type, how many values are present, how many are missing, and how many distinct values there are. Sheet row counts are below the column table. `Customer ID` is stored as a float because missing IDs are `NaN`

In [3]:
n_rows = len(df)

sheet_counts = pd.DataFrame(
    {"sheet": list(sheets.keys()), "rows": [len(sheet) for sheet in sheets.values()]}
)
sheet_counts.loc[len(sheet_counts)] = ["Total", sheet_counts["rows"].sum()]

profile = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).to_numpy(),
    "non_null": df.notna().sum().to_numpy(),
    "missing": df.isna().sum().to_numpy(),
    "missing_pct": (df.isna().mean().to_numpy() * 100).round(2),
    "n_unique": [df[col].nunique(dropna=True) for col in df.columns],
})

print(f"Total rows: {n_rows:,}")
print("\nRows per sheet")
print(sheet_counts.to_string(index=False))
print("\nColumn profile")
print(profile.to_string(index=False))

sheet_counts

Total rows: 1,067,371

Rows per sheet
         sheet    rows
Year 2009-2010  525461
Year 2010-2011  541910
         Total 1067371

Column profile
     column          dtype  non_null  missing  missing_pct  n_unique
    Invoice         object   1067371        0         0.00     53628
  StockCode         object   1067371        0         0.00      5305
Description         object   1062989     4382         0.41      5698
   Quantity          int64   1067371        0         0.00      1057
InvoiceDate datetime64[us]   1067371        0         0.00     47635
      Price        float64   1067371        0         0.00      2807
Customer ID        float64    824364   243007        22.77      5942
    Country            str   1067371        0         0.00        43


,sheet,rows
0,Year 2009-2010,525461
1,Year 2010-2011,541910
2,Total,1067371


## DE2 cleaning log

Each row is counted once, under the first rule that removes it.

1. **Cancellations.** Invoice starts with `C`. These are voids, not purchases.
2. **Non-positive quantity.** `Quantity <= 0`. These are returns and adjustments.
3. **Non-positive price.** `Price <= 0`. A zero or negative price is not a purchase.
4. **Non-product codes.** Postage, fees, discounts, samples, adjustments, tests, and vouchers (`POST`, `DOT`, `M`, `D`, `S`, `BANK CHARGES`, `AMAZONFEE`, and similar).
   A letter on the end of a numeric code is a variant, so those rows stay. `SP1002` stays too.
5. **Missing customer ID.** The matrix is one row per customer, so a line with no ID cannot be used.


In [ ]:
clean = df.copy()
clean["Invoice"] = clean["Invoice"].astype(str).str.strip()
clean["StockCode"] = clean["StockCode"].astype(str).str.strip()

NON_PRODUCT_CODES = {
    "POST",       
    "DOT",        
    "M", "m",    
    "C2",        
    "D",          
    "S",          
    "BANK CHARGES",
    "AMAZONFEE",
    "B",          
    "ADJUST", "ADJUST2",
    "CRUK",     
    "TEST001", "TEST002",
    "PADS",       
}

def non_product(codes):
    return codes.isin(NON_PRODUCT_CODES) | codes.str.startswith("gift_0001_")

steps = [
    (
        "Cancellations",
        "Invoice starts with C",
        lambda d: d["Invoice"].str.startswith("C"),
    ),
    (
        "Non-positive quantity",
        "Quantity <= 0",
        lambda d: d["Quantity"] <= 0,
    ),
    (
        "Non-positive price",
        "Price <= 0",
        lambda d: d["Price"] <= 0,
    ),
    (
        "Non-product codes",
        "Postage, fees, discounts, samples, adjustments, tests, vouchers",
        lambda d: non_product(d["StockCode"]),
    ),
    (
        "Missing customer ID",
        "Customer ID is null",
        lambda d: d["Customer ID"].isna(),
    ),
]

starting_rows = len(clean)
log_rows = []
for rule, reason, mask_fn in steps:
    drop = mask_fn(clean)
    removed = int(drop.sum())
    clean = clean.loc[~drop].copy()
    log_rows.append({
        "rule": rule,
        "reason": reason,
        "rows_removed": removed,
        "rows_remaining": len(clean),
    })

cleaning_log = pd.DataFrame(log_rows)
cleaning_log.loc[len(cleaning_log)] = {
    "rule": "Total removed",
    "reason": "",
    "rows_removed": int(cleaning_log["rows_removed"].sum()),
    "rows_remaining": len(clean),
}

kept_codes = sorted(clean.loc[~clean["StockCode"].str.match(r"^\d"), "StockCode"].unique())
print(f"Started with {starting_rows:,} rows")
print(f"Clean rows: {len(clean):,} ({len(clean) / starting_rows:.1%} kept)")
print("Non-numeric stock codes still in the clean data:", kept_codes)
print(
    "Checks:",
    "cancellations", int(clean["Invoice"].str.startswith("C").sum()),
    "qty<=0", int((clean["Quantity"] <= 0).sum()),
    "price<=0", int((clean["Price"] <= 0).sum()),
    "non-product", int(non_product(clean["StockCode"]).sum()),
    "missing id", int(clean["Customer ID"].isna().sum()),
)

cleaning_log

Started with 1,067,371 rows
Clean rows: 802,634 (75.2% kept)
Non-numeric stock codes still in the clean data: ['SP1002']
Checks: cancellations 0 qty<=0 0 price<=0 0 non-product 0 missing id 0


,rule,reason,rows_removed,rows_remaining
0,Cancellations,Invoice starts with C,19494,1047877
1,Non-positive quantity,Quantity <= 0,3457,1044420
2,Non-positive price,Price <= 0,2750,1041670
3,Non-product codes,"Postage, fees, discounts, samples, adjustments...",4666,1037004
4,Missing customer ID,Customer ID is null,234370,802634
5,Total removed,,264737,802634
